# As-of Joins on GPU

As-of joins are a foundational building block in time-series analytics, powering many of the most data-intensive workflows in finance and beyond.
Whether that be for trade-quote alignment, order book reconstruction, signal and feature alignment, protfolio valuation/risk snapshots, or IoT and sensor data synchronization, the KDB-X GPU Acceleration utilizes `.gpu.aj` to parallelize binary searches across every symbol simultaneously, using thousands of GPU cores.

In this tutorial, we demonstrate performance differences of joining trade and quote data on GPU vs CPU.

The core function we expose is `.gpu.aj` which is called as follows:
```q
.gpu.aj[c;t1;t2]
```
where
- `t1` is a table
- `t2` is a table
- `c` is a symbol list of n column names, common to `t1` and `t2`, and of matching type
- column `c[n]` is of a sortable type (typically time)

It's important to note that:
- columns in the list `c` may be mapped to the gpu for improved performance
- the list of values `c` can be of length 2 at most
- if `c` is of length 2, the only attribute supported on c[0] is the grouped attribute ``g#`

Data can be transferred between CPU and GPU as follows:
```q
.gpu.to t
```

To map only specific columns to the gpu (`time` and `sym` here), use `.gpu.xto`:
```q
.gpu.xto[`time`sym] t
```

For asof joins (`aj`):
- API automatically transfers to and from for CPU resident tables
- Better performance will be achieved by leaving target columns resident on GPU and using append for updates
- Limited to x2 columns

## Setup

This notebook is compatible with Linux, Mac, and Windows via WSL.

This notebook is designed to be run as a Python notebook, but the same functionality can be applied directly in a CLI with KDB-X (given it's running on a KDB-X CUDA application environment).
If running `q` from the terminal, just copy the `q` code and paste into your terminal.

We first want to download the required datasets and generate a database of trade and quote data.
We can configure the amount of data downloaded as a single day of NYSE TAQ files contain a huge amount of data - more on that [here](??).

Run `src/genHDB.sh` with the intended data configuration in order to download this data and generate an HDB using modified KX TAQ scripts (`src/tq.q`).
Build the HDB on an intended storage path, then [mount that directory to the Docker container at runtime](??)

In [2]:
import pykx as kx
import pandas as pd
import matplotlib.pyplot as plt
kx.q("\\s 48")

pykx.Identity(pykx.q('::'))

In [3]:
kx.Identity(kx.q('::'))

pykx.Identity(pykx.q('::'))

Something to note here: in order to load our GPU module, we are using
```q
.gpu:use`gpu
```
This is because of how the Docker container is setting `.Q.m.SP`, which points to `/app/.kx/mod/kx` - this will not always be the case as it can sometimes point to `$QHOME/.kx/mod`, in which case
```q
.gpu:use`kx.gpu
```
should be called instead. More on this [here](https://code.kx.com/kdb-x/modules/module-framework/quickstart.html#search-path)

In [4]:
%%q
// check .Q.m.SP and load module accordingly
first .Q.m.SP
.gpu:use`gpu

/app/.kx/mod/kx


In [5]:
%%q
\l /app/example/HDB/data
\c 1000 1000
show (count trade;count quote)

569917 5007138


In [6]:
%%q
t:get `trade
q:get `quote
show (meta t;meta q)

(+(,`c)!,`date`Time`Exchange`Symbol`SaleCondition`TradeVolume`TradePrice`TradeStopStockIndicator`TradeCorrectionIndicator`SequenceNumber`TradeId`SourceofTrade`TradeReportingFacility`ParticipantTimestamp`TradeReportingFacilityTRFTimestamp`TradeThroughExemptIndicator)!+`t`f`a!("dncssiebhiCcbnnb";````````````````;````p````````````)
(+(,`c)!,`date`Time`Exchange`Symbol`Bid_Price`Bid_Size`Offer_Price`Offer_Size`Quote_Condition`Sequence_Number`National_BBO_Ind`FINRA_BBO_Indicator`FINRA_ADF_MPID_Indicator`Quote_Cancel_Correction`Source_Of_Quote`Retail_Interest_Indicator`Short_Sale_Restriction_Indicator`LULD_BBO_Indicator`SIP_Generated_Message_Identifier`National_BBO_LULD_Indicator`Participant_Timestamp`FINRA_ADF_Timestamp`FINRA_ADF_Market_Participant_Quote_Indicator`Security_Status_Indicator)!+`t`f`a!("dncsfificiccccccccccnncc";````````````````````````;````p````````````````````)


In [7]:
%%q
\t tt:select from t where i > -1

41


In [9]:
%%q
\t qt:select Time, `g#Symbol, Bid_Price from q where i > -1

26


In [ ]:
%%q
\t T:.gpu.xto[`Time`Symbol] t

: 

: 

: 

In [1]:
%%q
\t tt:select from t where i > -1
\t qt:select Time,`g#Symbol,Bid_Price from q where i > -1
\t T:.gpu.xto[`Time`Symbol] t
\t Q:.gpu.xto[`Time`Symbol] q

UsageError: Cell magic `%%q` not found.


In [ ]:
%%q
\t:5 aj[`sym`time;t;q]
\t:5 .gpu.aj[`sym`time;t;Q]
\t:5 .gpu.aj[`sym`time;T;Q]